<a href="https://colab.research.google.com/github/Supan-Roy/financial-scam-analysis/blob/main/notebooks/06_transformer_based_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Transformer-Based Financial Scam Narrative Classification

## 1. Research Objective

This notebook investigates pretrained Transformer-based language models for multi-level financial scam narrative analysis.

The primary objective is to investigate whether contextual language representations can improve classification performance and cross-generation robustness compared with the traditional TF-IDF-based machine-learning baselines established in the previous experiments.

### Research Tasks

1. Scam subtype classification
2. Fraudulent intent classification
3. Social-engineering tactic recognition

### Experimental Approach

- Begin with a pretrained Transformer model as the initial baseline.
- Fine-tune the model using the existing synthetic datasets.
- Evaluate performance on both V3.1 and LLM-authored test distributions.
- Compare Transformer results against the traditional machine-learning baselines.
- Investigate classification errors and model interpretability.

### Initial Model

DistilBERT will be investigated as the starting pretrained Transformer architecture.

The model choice and architecture may be revised based on experimental findings.

### Experimental Considerations

- Preserve the established dataset partitions.
- Avoid train-test leakage.
- Maintain consistent evaluation metrics.
- Distinguish synthetic cross-generation performance from real-world generalization.
- Document model configurations, hyperparameters and limitations.


In [1]:
# Authenticate with GitHub from the Colab runtime

from google.colab import auth
auth.authenticate_user()

In [2]:
# Clone the private research repository using Colab Secrets

from google.colab import userdata
import subprocess

github_token = userdata.get("GITHUB_TOKEN")

repo_url = (
    "https://x-access-token:"
    + github_token
    + "@github.com/Supan-Roy/financial-scam-analysis.git"
)

subprocess.run(
    ["git", "clone", repo_url],
    check=True
)

print("Private repository cloned successfully.")

Private repository cloned successfully.


In [3]:
# Verify the processed research datasets

from pathlib import Path

PROJECT_ROOT = Path("/content/financial-scam-analysis")

V31_PATH = PROJECT_ROOT / "data/processed/synthetic_financial_scams_v3_1_processed.csv"
LLM_PATH = PROJECT_ROOT / "data/processed/llm_authored_financial_scams_processed.csv"

print("V3.1 dataset exists:", V31_PATH.exists())
print("LLM-authored dataset exists:", LLM_PATH.exists())

if V31_PATH.exists() and LLM_PATH.exists():
    print("\nBoth datasets are available.")
else:
    print("\nWarning: One or more datasets are missing.")

V3.1 dataset exists: True
LLM-authored dataset exists: True

Both datasets are available.


In [4]:
# Load both datasets and inspect their existing structure

import pandas as pd
import numpy as np
import random
import torch

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

df_v31 = pd.read_csv(V31_PATH)
df_llm = pd.read_csv(LLM_PATH)

print("V3.1 dataset:", df_v31.shape)
print("LLM-authored dataset:", df_llm.shape)

print("\nV3.1 partition distribution:")
print(df_v31["split_partition"].value_counts())

print("\nLLM-authored partition distribution:")
print(df_llm["split_partition"].value_counts())

V3.1 dataset: (7800, 36)
LLM-authored dataset: (2730, 45)

V3.1 partition distribution:
split_partition
train    6240
val       780
test      780
Name: count, dtype: int64

LLM-authored partition distribution:
split_partition
train    2184
val       273
test      273
Name: count, dtype: int64


In [17]:
# Inspect target labels and their distributions

target_columns = [
    "scam_subtype",
    "fraudulent_intent",
    "social_engineering_tactics"
]

for dataset_name, df in [
    ("V3.1", df_v31),
    ("LLM-authored", df_llm)
]:
    print(f"\n{'=' * 60}")
    print(dataset_name)

    for column in target_columns:
        print(f"\n{column}")
        print("Missing:", df[column].isna().sum())
        print("Unique labels:", df[column].nunique())

        if column != "social_engineering_tactics":
            print(df[column].value_counts())


V3.1

scam_subtype
Missing: 0
Unique labels: 13
scam_subtype
account_verification              600
card_transaction_alert            600
refund_chargeback                 600
invoice_payment_request           600
loan_offer                        600
investment_crypto                 600
prize_reward                      600
tax_payment                       600
job_task_commission               600
romance_financial_manipulation    600
family_emergency_request          600
recovery_support                  600
charity_donation                  600
Name: count, dtype: int64

fraudulent_intent
Missing: 0
Unique labels: 5
fraudulent_intent
induce_payment_or_transfer                  3684
obtain_personal_or_financial_information    1792
obtain_credentials_or_codes                 1160
gain_account_or_device_access                582
induce_fraudulent_investment                 582
Name: count, dtype: int64

social_engineering_tactics
Missing: 0
Unique labels: 223

LLM-authored

scam_subt

In [12]:
# Mount Google Drive for accessing large research datasets

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [14]:
# Load binary classification partitions from Google Drive

BINARY_DATA_PATH = (
    Path("/content/drive/MyDrive/financial-scam-analysis/data/splits")
)

TRAIN_PATH = BINARY_DATA_PATH / "train.csv"
VAL_PATH = BINARY_DATA_PATH / "validation.csv"
TEST_PATH = BINARY_DATA_PATH / "test.csv"

train_binary = pd.read_csv(TRAIN_PATH, low_memory=False)
val_binary = pd.read_csv(VAL_PATH, low_memory=False)
test_binary = pd.read_csv(TEST_PATH, low_memory=False)

print(train_binary.shape)
print(val_binary.shape)
print(test_binary.shape)

(60957, 32)
(7618, 32)
(7618, 32)


In [15]:
# Identify the column with mixed data types

mixed_column = train_binary.columns[13]

print("Column index:", 13)
print("Column name:", mixed_column)
print("\nData types found:")
print(train_binary[mixed_column].apply(type).value_counts())

print("\nSample values:")
print(train_binary[mixed_column].dropna().head(10).tolist())

Column index: 13
Column name: scam_subtype

Data types found:
scam_subtype
<class 'float'>    49866
<class 'str'>      11091
Name: count, dtype: int64

Sample values:
['financial_scam', 'financial_scam', 'financial_scam', 'financial_scam', 'financial_scam', 'financial_scam', 'financial_scam', 'financial_scam', 'financial_scam', 'financial_scam']


In [16]:
# Verify binary labels and required text columns

for name, data in [
    ("Train", train_binary),
    ("Validation", val_binary),
    ("Test", test_binary)
]:
    print(f"\n{name}")
    print("Label distribution:")
    print(data["label"].value_counts().sort_index())
    print("Missing labels:", data["label"].isna().sum())
    print("Missing text:", data["normalized_text"].isna().sum())
    print("Empty text:", data["normalized_text"].fillna("").str.strip().eq("").sum())


Train
Label distribution:
label
0    49866
1    11091
Name: count, dtype: int64
Missing labels: 0
Missing text: 0
Empty text: 0

Validation
Label distribution:
label
0    6233
1    1385
Name: count, dtype: int64
Missing labels: 0
Missing text: 0
Empty text: 0

Test
Label distribution:
label
0    6233
1    1385
Name: count, dtype: int64
Missing labels: 0
Missing text: 0
Empty text: 0


In [5]:
# Verify PyTorch and available GPU hardware

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("No GPU detected. Enable a GPU through Runtime > Change runtime type.")

PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
CUDA version: 13.0


In [7]:
# Install Hugging Face Transformer and evaluation dependencies

%pip install -q transformers datasets accelerate evaluate scikit-learn

In [8]:
# Verify Transformer dependencies

import torch
import transformers
import datasets
import accelerate
import sklearn

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("Scikit-learn:", sklearn.__version__)
print("GPU available:", torch.cuda.is_available())

PyTorch: 2.14.1+cu130
Transformers: 5.22.1
Datasets: 4.8.5
Accelerate: 1.15.0
Scikit-learn: 1.6.1
GPU available: True


In [18]:
# Verify dataset availability and Transformer components

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

required_variables = [
    "train_binary",
    "val_binary",
    "test_binary",
    "df_v31",
    "df_llm"
]

for name in required_variables:
    print(f"{name}: {'Available' if name in globals() else 'Not loaded'}")

print("\nTransformer components imported successfully.")
print("GPU:", torch.cuda.get_device_name(0))

train_binary: Available
val_binary: Available
test_binary: Available
df_v31: Available
df_llm: Available

Transformer components imported successfully.
GPU: Tesla T4


In [19]:
# Load the pretrained DistilBERT tokenizer

from transformers import AutoTokenizer

MODEL_NAME = "distilbert/distilbert-base-uncased"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully.")
print("Model:", MODEL_NAME)
print("Vocabulary size:", tokenizer.vocab_size)
print("Maximum sequence length:", MAX_LENGTH)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer loaded successfully.
Model: distilbert/distilbert-base-uncased
Vocabulary size: 30522
Maximum sequence length: 256
